In [29]:
import pandas as pd 
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from faker import Faker
from datetime import datetime, timedelta


## 1

In [30]:
fake = Faker()
Faker.seed(42)
np.random.seed(42)
fake = Faker()

In [31]:
product_catalog = {
    "Laptop": (800, 2500),
    "Phone": (300, 1500),
    "Tablet": (200, 1200),
    "Headphones": (50, 500),
    "Monitor": (150, 1000),
    "Guitar" : (200,3000),
    "Bicycle" : (500,2500),
    "Scooter" : (700,1800)
}

records = []

for i in range(1, 201):
    product = np.random.choice(list(product_catalog.keys()))
    price_range = product_catalog[product]
    order_date = fake.date_time_this_year()
    
    if isinstance(order_date, datetime):
        order_date = order_date.strftime("%Y-%m-%d")
    
    records.append({
        "order_id": f'O{i:03}',
        "customer_id": f'C{i:03}', 
        "product_name": product,
        "unit_price": round(np.random.uniform(*price_range), 2),
        "quantity": np.random.randint(1, 50),
        "order_date": order_date,
        "shipping_country": fake.country()
    })
    
for i in range(3):
    n = np.random.randint(0,200)
    records[n]['product_name'] = ""

for i in range(3):
    n = np.random.randint(0,200)
    records[n]['quantity'] = records[n]['quantity'] * -1
    

for i in range(3):
    n = np.random.randint(0,200)
    records[n]['unit_price'] = records[n]['unit_price'] * -1
    
for i in range(3):
    n = np.random.randint(0,200)
    records[n]['order_date'] = 'not-a-date'
    
for i in range(3):
    n = np.random.randint(0,200)
    y = np.random.randint(2020,2026)
    m = np.random.randint(1,13)
    d = np.random.randint(32,100)
    records[n]['order_date'] = f"{y}-{m}-{d}"
    

for i in range(3):
    n = np.random.randint(0,200)
    m = np.random.randint(0,200)
    records[n]['order_id'] = records[m]['order_id']

for i in range(3):
    n = np.random.randint(0,200)
    records[n]['quantity'] = str(records[n]['quantity'])
    

for i in range(3):
    n = np.random.randint(0,200)
    records[n]['unit_price'] = str(records[n]['unit_price'])


In [32]:
def safe_int(value):
    try:
        return int(value)
    except (ValueError, TypeError):
        return None

def safe_float(value):
    try:
        return float(value)
    except (ValueError, TypeError):
        return None

def extract(raw_records):
    valid = []
    invalid = []
    order_ids = set()

    for record in raw_records:
        order_id = record.get('order_id')

        if order_id in order_ids:
            record['reason'] = 'duplicate_order_id'  
            invalid.append(record)
            continue
        order_ids.add(order_id)

        if not record.get('product_name') or record['product_name'] not in product_catalog:
            record['reason'] = 'invalid_product_name'
            invalid.append(record)
            continue

        unit_price = safe_float(record.get('unit_price'))
        if unit_price is None or unit_price < 0:
            record['reason'] = 'invalid_unit_price'
            invalid.append(record)
            continue
        record['unit_price'] = unit_price

        quantity = safe_int(record.get('quantity'))
        if quantity is None or quantity < 0:
            record['reason'] = 'invalid_quantity'
            invalid.append(record)
            continue
        record['quantity'] = quantity  

        try:
            datetime.strptime(str(record['order_date']), "%Y-%m-%d")
        except Exception:
            record['reason'] = 'invalid_date'
            invalid.append(record)
            continue

        valid.append(record)

    return valid, invalid
        
        
valid_records, invalid_records = extract(records)
    
print('Number of valid records: ', len(valid_records))
print('Number of invalid records: ', len(invalid_records))
            

Number of valid records:  182
Number of invalid records:  18


In [33]:
def transform(valid_records):
    import pandas as pd

    records = pd.DataFrame(valid_records)
    records['order_date'] = pd.to_datetime(records['order_date'])
    records['total_amount'] = records['quantity'] * records['unit_price']
    records['order_month'] = records['order_date'].dt.to_period('M')
    records['order_day_of_week'] = records['order_date'].dt.day_name()
    records['shipping_country'] = records['shipping_country'].str.title()
    records = records.drop_duplicates(subset=['order_id'], keep='first')

    records['amount_category'] = 'medium'
    records.loc[records['total_amount'] < 25, 'amount_category'] = 'small'
    records.loc[records['total_amount'] > 100, 'amount_category'] = 'large'

    return records

In [34]:
def load(df, path):
    import pandas as pd
    import os
    df.to_parquet(path)
    check_parquet = pd.read_parquet(path)
    if (df.dtypes == check_parquet.dtypes).all():
        print('The parquet file is created correctly!')
    else:
        os.remove(path)
        print('The problem arose!')

In [35]:
valid_records, invalid_records = extract(records)
transformed = transform(valid_records)        
load(transformed, 'records.parquet')

The parquet file is created correctly!


In [36]:
raw_count = len(records)
valid_count = len(valid_records)
transformed_count = len(transformed)

print("Raw records:", raw_count)
print("Valid records:", valid_count)
print("Rejected records:", len(invalid_records))
print("Transformed records:", transformed_count)

Raw records: 200
Valid records: 182
Rejected records: 18
Transformed records: 182


## 2

In [37]:
def extract(raw_records):
    extracted = []
    rejected = []

    for record in raw_records:
        if isinstance(record, dict):
            extracted.append(record)
        else:
            rejected.append({'record': record, 'reason': 'invalid_record_structure'})

    return extracted, rejected

In [38]:
def load_raw(records, path):
    import pandas as pd
    
    df = pd.DataFrame(records)
    df.to_parquet(path)

    print("Raw data loaded to data lake:", path)

In [39]:
def transform(path):
    import pandas as pd
    from datetime import datetime

    df = pd.read_parquet(path)

    df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
    df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')

    df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

    df = df[
        (df['product_name'].notna()) &
        (df['product_name'] != "") &
        (df['quantity'] > 0) &
        (df['unit_price'] > 0) &
        (df['order_date'].notna())
    ]

    df = df.drop_duplicates(subset='order_id')

    df['total_amount'] = df['quantity'] * df['unit_price']
    df['order_month'] = df['order_date'].dt.to_period('M')
    df['order_day_of_week'] = df['order_date'].dt.day_name()

    df['shipping_country'] = df['shipping_country'].str.title()

    df['amount_category'] = 'medium'
    df.loc[df['total_amount'] < 25, 'amount_category'] = 'small'
    df.loc[df['total_amount'] > 100, 'amount_category'] = 'large'

    return df

In [40]:
raw_count = len(records)

extracted, rejected = extract(records)

load_raw(extracted, "raw_orders.parquet")

clean_df = transform("raw_orders.parquet")

print("Raw records:", raw_count)
print("Extracted records:", len(extracted))
print("Rejected during extract:", len(rejected))
print("Clean records after transform:", len(clean_df))

Raw data loaded to data lake: raw_orders.parquet
Raw records: 200
Extracted records: 200
Rejected during extract: 0
Clean records after transform: 182


In the ETL pipeline, the data was checked early in the extract step. Because of that, bad records were removed before loading. After the cleaning and transform step, 182 records finally made it to the destination dataset.

In the ELT pipeline, all 200 records were first loaded into the raw data lake (raw_orders.parquet). The problems in the data were not checked at that time. Later, during the transform step, invalid rows were filtered out. After that cleaning process, 182 records remained in the final dataset.

So ETL stops bad data earlier, while ELT lets everything load first and fixes the problems later.

## 3

In [41]:
database = {"orders": [], "features": {}}

def order_service_write(order):
    database["orders"].append(order)
    

In [42]:
def feature_service_compute():
    from collections import defaultdict
    from datetime import datetime
    
    orders = database["orders"]
    
    customer_orders = defaultdict(list)

    for order in orders:
        customer_orders[order["customer_id"]].append(order)

    features = {}

    for customer_id, orders_list in customer_orders.items():
        amounts = [o["amount"] for o in orders_list]
        dates = [datetime.strptime(o["order_date"], "%Y-%m-%d") for o in orders_list]

        features[customer_id] = {
            "total_orders": len(orders_list),
            "average_amount": sum(amounts) / len(amounts),
            "last_order_date": max(dates).strftime("%Y-%m-%d")
        }

    database["features"] = features

In [43]:
def prediction_service_read(customer_id):
    return database["features"].get(customer_id, "No features found")

In [44]:
orders_db = []

def order_service_write(order):
    orders_db.append(order)

def order_service_get_orders(customer_id):
    return [o for o in orders_db if o["customer_id"] == customer_id]

In [45]:
def feature_service_get_features(customer_id):
    from datetime import datetime
    orders = order_service_get_orders(customer_id)

    if not orders:
        return None

    amounts = [o["amount"] for o in orders]
    dates = [datetime.strptime(o["order_date"], "%Y-%m-%d") for o in orders]

    features = {
        "total_orders": len(orders),
        "average_amount": sum(amounts) / len(amounts),
        "last_order_date": max(dates).strftime("%Y-%m-%d")
    }

    return features

In [46]:
def prediction_service_request(customer_id):

    features = feature_service_get_features(customer_id)

    if features is None:
        return "No data for customer"

    return features

In [47]:
class PubSubBroker:
    
    def __init__(self):
        self.topics = {}
        self.subscribers = {}

    def publish(self, topic, message):
        if topic not in self.topics:
            self.topics[topic] = []

        self.topics[topic].append(message)

        if topic in self.subscribers:
            for func in self.subscribers[topic]:
                func(message)

    def subscribe(self, topic, func):
        if topic not in self.subscribers:
            self.subscribers[topic] = []

        self.subscribers[topic].append(func)

    def get_latest(self, topic):
        if topic in self.topics and self.topics[topic]:
            return self.topics[topic][-1]
        return None

In [48]:
broker = PubSubBroker()

In [49]:
def order_service_publish(order):
    broker.publish("orders", order)

In [50]:
from datetime import datetime
from collections import defaultdict

customer_orders = defaultdict(list)

def feature_service_compute(order):

    customer_id = order["customer_id"]
    customer_orders[customer_id].append(order)

    orders = customer_orders[customer_id]

    amounts = [o["amount"] for o in orders]
    dates = [datetime.strptime(o["order_date"], "%Y-%m-%d") for o in orders]

    features = {
        "customer_id": customer_id,
        "total_orders": len(orders),
        "average_amount": sum(amounts) / len(amounts),
        "last_order_date": max(dates).strftime("%Y-%m-%d")
    }

    broker.publish("features", features)

In [51]:
broker.subscribe("orders", feature_service_compute)

In [52]:
def prediction_service_receive(features):
    print("Prediction service received features:")
    print(features)

In [53]:
broker.subscribe("features", prediction_service_receive)

In [54]:
sample_orders = transformed.head(20).to_dict(orient="records")

db_results = []
service_results = []
pubsub_results = []

print("\nChecking if outputs match across modes:\n")

all_match = True

for i, order in enumerate(sample_orders):

    order_dict = {
        "customer_id": order["customer_id"],
        "order_date": str(order["order_date"].date()),
        "amount": order["total_amount"]
    }

    orders_db.clear()
    order_service_write(order_dict)
    db_feature = prediction_service_request(order_dict["customer_id"])
    db_total = db_feature["total_orders"]

    orders_db.clear()
    order_service_write(order_dict)
    service_feature = prediction_service_request(order_dict["customer_id"])
    service_total = service_feature["total_orders"]

    customer_orders.clear()
    order_service_publish(order_dict)
    pubsub_total = len(customer_orders[order_dict["customer_id"]])

    match = (db_total == service_total == pubsub_total)

    if not match:
        all_match = False

    print(
        f"Order {i+1}: DB={db_total}, Services={service_total}, PubSub={pubsub_total} -> Match: {match}"
    )

print("\nFinal result:")
print("All three architectures produced matching results:", all_match)


Checking if outputs match across modes:

Prediction service received features:
{'customer_id': 'C001', 'total_orders': 1, 'average_amount': 31396.350000000002, 'last_order_date': '2026-02-12'}
Order 1: DB=1, Services=1, PubSub=1 -> Match: True
Prediction service received features:
{'customer_id': 'C002', 'total_orders': 1, 'average_amount': 20573.49, 'last_order_date': '2026-02-19'}
Order 2: DB=1, Services=1, PubSub=1 -> Match: True
Prediction service received features:
{'customer_id': 'C003', 'total_orders': 1, 'average_amount': 32008.410000000003, 'last_order_date': '2026-01-15'}
Order 3: DB=1, Services=1, PubSub=1 -> Match: True
Prediction service received features:
{'customer_id': 'C004', 'total_orders': 1, 'average_amount': 23733.0, 'last_order_date': '2026-01-07'}
Order 4: DB=1, Services=1, PubSub=1 -> Match: True
Prediction service received features:
{'customer_id': 'C005', 'total_orders': 1, 'average_amount': 32535.36, 'last_order_date': '2026-03-01'}
Order 5: DB=1, Services=1

## 4

In [55]:
transformed

,order_id,customer_id,product_name,unit_price,quantity,order_date,shipping_country,total_amount,order_month,order_day_of_week,amount_category
0,O001,C001,Bicycle,2093.09,15,2026-02-12,Anguilla,31396.35,2026-02,Thursday,large
1,O002,C002,Tablet,979.69,21,2026-02-19,Ecuador,20573.49,2026-02,Thursday,large
2,O003,C003,Bicycle,1391.67,23,2026-01-15,Saint Pierre And Miquelon,32008.41,2026-01,Thursday,large
3,O004,C004,Tablet,659.25,36,2026-01-07,Saint Vincent And The Grenadines,23733.00,2026-01,Wednesday,large
4,O005,C005,Scooter,1478.88,22,2026-03-01,Benin,32535.36,2026-03,Sunday,large
...,...,...,...,...,...,...,...,...,...,...,...
177,O195,C195,Monitor,448.28,18,2026-01-11,Kenya,8069.04,2026-01,Sunday,large
178,O196,C196,Bicycle,785.98,10,2026-02-06,Japan,7859.80,2026-02,Friday,large
179,O197,C197,Phone,1041.86,38,2026-02-07,Bouvet Island (Bouvetoya),39590.68,2026-02,Saturday,large
180,O198,C198,Bicycle,1352.18,40,2026-01-06,Turks And Caicos Islands,54087.20,2026-01,Tuesday,large


In [56]:
total_orders_per_day = transformed.groupby(transformed['order_date'].dt.date).size()
total_orders_per_day

order_date
2026-01-01    3
2026-01-02    3
2026-01-04    4
2026-01-05    5
2026-01-06    5
2026-01-07    7
2026-01-08    1
2026-01-09    2
2026-01-11    3
2026-01-13    2
2026-01-14    1
2026-01-15    6
2026-01-16    4
2026-01-17    5
2026-01-18    9
2026-01-19    4
2026-01-20    1
2026-01-21    1
2026-01-23    2
2026-01-24    1
2026-01-25    2
2026-01-26    2
2026-01-27    2
2026-01-28    2
2026-01-29    4
2026-01-30    1
2026-01-31    5
2026-02-01    1
2026-02-03    1
2026-02-04    3
2026-02-05    1
2026-02-06    2
2026-02-07    4
2026-02-08    4
2026-02-09    1
2026-02-10    2
2026-02-11    2
2026-02-12    6
2026-02-13    5
2026-02-14    2
2026-02-15    5
2026-02-16    3
2026-02-17    5
2026-02-18    6
2026-02-19    2
2026-02-20    4
2026-02-21    2
2026-02-23    3
2026-02-24    3
2026-02-25    1
2026-02-26    1
2026-02-27    5
2026-02-28    1
2026-03-01    4
2026-03-03    3
2026-03-04    2
2026-03-05    2
2026-03-06    1
2026-03-07    3
2026-03-08    5
dtype: int64

In [57]:
total_revenue_per_day = transformed.groupby(transformed['order_date'].dt.date)['total_amount'].sum()
total_revenue_per_day

order_date
2026-01-01    143332.17
2026-01-02    120939.70
2026-01-04     38483.93
2026-01-05     73476.41
2026-01-06    159740.88
2026-01-07    198521.62
2026-01-08     47637.50
2026-01-09     36852.88
2026-01-11     64398.25
2026-01-13     48297.44
2026-01-14     17375.69
2026-01-15    214963.90
2026-01-16    206779.51
2026-01-17     75586.93
2026-01-18    257110.01
2026-01-19     91700.79
2026-01-20     47448.96
2026-01-21     21278.76
2026-01-23     41468.77
2026-01-24     39935.52
2026-01-25     48691.12
2026-01-26     71820.81
2026-01-27     55492.40
2026-01-28     83706.87
2026-01-29     77129.16
2026-01-30     77973.12
2026-01-31    196620.90
2026-02-01     11274.64
2026-02-03     22571.64
2026-02-04     48972.58
2026-02-05     57207.60
2026-02-06     24201.56
2026-02-07     86027.66
2026-02-08     86774.08
2026-02-09     42874.48
2026-02-10     29885.43
2026-02-11     48529.96
2026-02-12    137140.11
2026-02-13    148340.44
2026-02-14     40097.02
2026-02-15     69278.38
2026-

In [58]:
avg_order_size_per_day= round(transformed.groupby(transformed['order_date'].dt.date)['quantity'].mean())
avg_order_size_per_day

order_date
2026-01-01    34.0
2026-01-02    35.0
2026-01-04    24.0
2026-01-05    25.0
2026-01-06    25.0
2026-01-07    30.0
2026-01-08    37.0
2026-01-09    19.0
2026-01-11    26.0
2026-01-13    22.0
2026-01-14    19.0
2026-01-15    34.0
2026-01-16    32.0
2026-01-17    19.0
2026-01-18    35.0
2026-01-19    23.0
2026-01-20    32.0
2026-01-21    12.0
2026-01-23    30.0
2026-01-24    36.0
2026-01-25    22.0
2026-01-26    29.0
2026-01-27    30.0
2026-01-28    34.0
2026-01-29    18.0
2026-01-30    36.0
2026-01-31    28.0
2026-02-01    37.0
2026-02-03    26.0
2026-02-04    21.0
2026-02-05    24.0
2026-02-06    13.0
2026-02-07    17.0
2026-02-08    26.0
2026-02-09    44.0
2026-02-10    20.0
2026-02-11    26.0
2026-02-12    19.0
2026-02-13    25.0
2026-02-14    12.0
2026-02-15    21.0
2026-02-16    23.0
2026-02-17    33.0
2026-02-18    32.0
2026-02-19    14.0
2026-02-20    30.0
2026-02-21    28.0
2026-02-23    21.0
2026-02-24    10.0
2026-02-25    31.0
2026-02-26    21.0
2026-02-27    13.0
2

In [59]:
unique_customers_per_day = transformed.groupby(transformed['order_date'].dt.date)['customer_id'].nunique()
unique_customers_per_day


order_date
2026-01-01    3
2026-01-02    3
2026-01-04    4
2026-01-05    5
2026-01-06    5
2026-01-07    7
2026-01-08    1
2026-01-09    2
2026-01-11    3
2026-01-13    2
2026-01-14    1
2026-01-15    6
2026-01-16    4
2026-01-17    5
2026-01-18    9
2026-01-19    4
2026-01-20    1
2026-01-21    1
2026-01-23    2
2026-01-24    1
2026-01-25    2
2026-01-26    2
2026-01-27    2
2026-01-28    2
2026-01-29    4
2026-01-30    1
2026-01-31    5
2026-02-01    1
2026-02-03    1
2026-02-04    3
2026-02-05    1
2026-02-06    2
2026-02-07    4
2026-02-08    4
2026-02-09    1
2026-02-10    2
2026-02-11    2
2026-02-12    6
2026-02-13    5
2026-02-14    2
2026-02-15    5
2026-02-16    3
2026-02-17    5
2026-02-18    6
2026-02-19    2
2026-02-20    4
2026-02-21    2
2026-02-23    3
2026-02-24    3
2026-02-25    1
2026-02-26    1
2026-02-27    5
2026-02-28    1
2026-03-01    4
2026-03-03    3
2026-03-04    2
2026-03-05    2
2026-03-06    1
2026-03-07    3
2026-03-08    5
Name: customer_id, dtype: int

In [60]:
daily_product_revenue = (
    transformed
    .groupby([transformed['order_date'].dt.date, 'product_name'])
    ['total_amount']
    .sum()
    .reset_index()
)
top_product_per_day = daily_product_revenue.loc[
    daily_product_revenue.groupby('order_date')['total_amount'].idxmax()
].reset_index(drop=True)
top_product_per_day

,order_date,product_name,total_amount
0,2026-01-01,Bicycle,75960.80
1,2026-01-02,Bicycle,78913.80
2,2026-01-04,Phone,17946.00
3,2026-01-05,Bicycle,38273.89
4,2026-01-06,Laptop,66895.68
5,2026-01-07,Scooter,76012.74
6,2026-01-08,Scooter,47637.50
7,2026-01-09,Laptop,36150.00
8,2026-01-11,Laptop,44165.88
9,2026-01-13,Laptop,37348.40


In [61]:
def compute_daily_aggregates(df):

    df['order_date'] = pd.to_datetime(df['order_date'])

    total_orders_per_day = df.groupby(df['order_date'].dt.date).size().rename("total_orders")


    avg_order_size_per_day = df.groupby(df['order_date'].dt.date)['quantity'].mean().round().rename("avg_order_size")

    unique_customers_per_day = df.groupby(df['order_date'].dt.date)['customer_id'].nunique().rename("unique_customers")

    daily_product_revenue = (
        df.groupby([df['order_date'].dt.date, 'product_name'])['total_amount']
        .sum()
        .reset_index()
    )
    top_product_per_day = daily_product_revenue.loc[
        daily_product_revenue.groupby('order_date')['total_amount'].idxmax()
    ].reset_index(drop=True)
    top_product_per_day = top_product_per_day.rename(columns={'product_name': 'top_product', 'total_amount': 'top_product_revenue'})

    daily_aggregates = pd.concat(
        [total_orders_per_day, total_revenue_per_day, avg_order_size_per_day, unique_customers_per_day],
        axis=1
    ).reset_index().rename(columns={'order_date': 'date'})

    daily_aggregates = daily_aggregates.merge(top_product_per_day[['order_date', 'top_product', 'top_product_revenue']],
                                              left_on='date', right_on='order_date', how='left').drop(columns='order_date')

    return daily_aggregates

In [62]:
print(compute_daily_aggregates(transformed))

          date  total_orders  total_amount  avg_order_size  unique_customers  \
0   2026-01-01             3     143332.17            34.0                 3   
1   2026-01-02             3     120939.70            35.0                 3   
2   2026-01-04             4      38483.93            24.0                 4   
3   2026-01-05             5      73476.41            25.0                 5   
4   2026-01-06             5     159740.88            25.0                 5   
5   2026-01-07             7     198521.62            30.0                 7   
6   2026-01-08             1      47637.50            37.0                 1   
7   2026-01-09             2      36852.88            19.0                 2   
8   2026-01-11             3      64398.25            26.0                 3   
9   2026-01-13             2      48297.44            22.0                 2   
10  2026-01-14             1      17375.69            19.0                 1   
11  2026-01-15             6     214963.

In [63]:
from collections import deque, Counter

class StreamProcessor:
    def __init__(self, window_size=50):
        self.total_orders = 0
        self.total_revenue = 0.0
        self.unique_customers = set()
        self.window_size = window_size
        
        self.recent_quantities = deque(maxlen=window_size)
        self.recent_products = deque(maxlen=window_size)
    
    def process_order(self, order):

        self.total_orders += 1
        self.total_revenue += order['total_amount']
        self.unique_customers.add(order['customer_id'])
        

        self.recent_quantities.append(order['quantity'])
        self.recent_products.append(order['product_name'])
        

        if self.total_orders % self.window_size == 0:
            self.print_stats()
    
    def rolling_avg_order_size(self):
        if self.recent_quantities:
            return sum(self.recent_quantities) / len(self.recent_quantities)
        return 0
    
    def most_popular_product(self):
        if self.recent_products:
            counter = Counter(self.recent_products)
            return counter.most_common(1)[0][0]
        return None
    
    def print_stats(self):
        print(f"Streaming stats after {self.total_orders} orders:")
        print(f"Running total orders: {self.total_orders}")
        print(f"Running total revenue: ${self.total_revenue:.2f}")
        print(f"Running unique customers: {len(self.unique_customers)}")
        print(f"Windowed average order size (last {self.window_size} orders): {self.rolling_avg_order_size():.2f}")
        print(f"Most popular product in last {self.window_size} orders: {self.most_popular_product()}")
        print("\n")


stream_processor = StreamProcessor(window_size=50)


for _, order in transformed.iterrows():
    stream_processor.process_order(order.to_dict())

Streaming stats after 50 orders:
Running total orders: 50
Running total revenue: $1314611.52
Running unique customers: 50
Windowed average order size (last 50 orders): 25.34
Most popular product in last 50 orders: Bicycle


Streaming stats after 100 orders:
Running total orders: 100
Running total revenue: $2748647.27
Running unique customers: 100
Windowed average order size (last 50 orders): 25.12
Most popular product in last 50 orders: Scooter


Streaming stats after 150 orders:
Running total orders: 150
Running total revenue: $4045123.33
Running unique customers: 150
Windowed average order size (last 50 orders): 25.12
Most popular product in last 50 orders: Headphones




When we compare cumulative totals, like total orders or total revenue, both windowed streaming and daily batch computations generally match because they are aggregating the same data.Windowed streaming statistics often differ from daily batch statistics in specific metrics, such as averages, counts within a moving window, or recency-based features. This happens because streaming metrics focus on a limited, recent subset of data, whereas batch metrics summarize all historical data.

Batch statistics provide a complete view of a customer’s long term behavior, capturing overall trends, totals and averages. They are useful for predicting long term value or historical patterns.

Streaming statistics capture short term, recent behavior, which can detect sudden changes, spikes in activity, or recent trends that might not be reflected in historical aggregates. They are useful for real-time decisions, such as targeting customers for flash sales or predicting immediate engagement.

## 5

In [64]:
import pandas as pd
from collections import Counter
from datetime import datetime

sample_customers = transformed['customer_id'].sample(5, random_state=42).tolist()

customer_features = {}

for cid in sample_customers:
    cust_orders = transformed[transformed['customer_id'] == cid].sort_values('order_date')
    
    total_lifetime_orders = len(cust_orders)
    avg_order_amount = cust_orders['total_amount'].mean()
    days_since_first_order = (datetime.now() - cust_orders['order_date'].min()).days
    most_purchased_category = cust_orders['product_name'].mode()[0]
    
    batch_features = {
        'total_lifetime_orders': total_lifetime_orders,
        'avg_order_amount': avg_order_amount,
        'days_since_first_order': days_since_first_order,
        'most_purchased_category': most_purchased_category
    }
    
    last_orders = cust_orders.tail(10)
    recent_order_count = len(last_orders)
    recent_avg_amount = last_orders['total_amount'].mean()
    seconds_since_last_order = (datetime.now() - last_orders['order_date'].max()).total_seconds()
    recent_top_category = last_orders['product_name'].mode()[0]
    
    streaming_features = {
        'recent_order_count': recent_order_count,
        'recent_avg_amount': recent_avg_amount,
        'seconds_since_last_order': seconds_since_last_order,
        'recent_top_category': recent_top_category
    }
    
    combined = {**batch_features, **streaming_features}
    customer_features[cid] = combined

for cid, feats in customer_features.items():
    print(f"Customer {cid}:")
    for k, v in feats.items():
        print(f"  {k}: {v}")
    print()

Customer C020:
  total_lifetime_orders: 1
  avg_order_amount: 39935.52
  days_since_first_order: 43
  most_purchased_category: Tablet
  recent_order_count: 1
  recent_avg_amount: 39935.52
  seconds_since_last_order: 3792367.849772
  recent_top_category: Tablet

Customer C046:
  total_lifetime_orders: 1
  avg_order_amount: 9383.14
  days_since_first_order: 13
  most_purchased_category: Bicycle
  recent_order_count: 1
  recent_avg_amount: 9383.14
  seconds_since_last_order: 1200367.852649
  recent_top_category: Bicycle

Customer C172:
  total_lifetime_orders: 1
  avg_order_amount: 101700.91
  days_since_first_order: 4
  most_purchased_category: Guitar
  recent_order_count: 1
  recent_avg_amount: 101700.91
  seconds_since_last_order: 422767.855182
  recent_top_category: Guitar

Customer C107:
  total_lifetime_orders: 1
  avg_order_amount: 94295.63
  days_since_first_order: 19
  most_purchased_category: Guitar
  recent_order_count: 1
  recent_avg_amount: 94295.63
  seconds_since_last_order

Machine learning models perform best when they have access to both historical trends (batch features) and recent behavior (stream features). Batch features provide long-term patterns. Stream features capture real-time dynamics.

Example where batch features alone are insufficient: Predicting whether a customer will make a purchase today. Historical averages may suggest typical behavior, but they cannot detect a sudden surge in interest or a flash sale that just started. Without stream features, the model would miss these short-term signals.

Example where stream features alone are insufficient: Predicting a customer’s lifetime value or long-term churn risk. Looking only at the last 50 orders may misrepresent a generally loyal or high-spending customer, because short-term behavior does not reflect their overall patterns. Batch features are necessary to capture long-term trends.